In [0]:
print("-----GIVEN DATAFRAME-----\n\n\n")

# First DataFrame - df (teams)
data = [
    ("India",),
    ("Pakistan",),
    ("SriLanka",)
]

columns = ["teams"]

df = spark.createDataFrame(data, columns)
df.show()


print("-----REQUIRED DATAFRAME-----\n\n\n")

# Second DataFrame - df1 (matches)
data1 = [
    ("India Vs Pakistan",),
    ("India Vs SriLanka",),
    ("Pakistan Vs SriLanka",)
]

columns1 = ["matches"]

df1 = spark.createDataFrame(data1, columns1)
df1.show()

print("-----DSL SOLUTION-----\n\n\n")

from pyspark.sql.functions import *


##Using cartesian join

redf = df.withColumnRenamed("teams", "teams1")


joindf = df.join(redf,df.teams<redf.teams1,"cross")
joindf.show()



concatdf = joindf.selectExpr("concat(teams,' Vs ',teams1) as matches")
concatdf.show()


##Self-join with bounded row numbers

numdf = df.withColumn("id", monotonically_increasing_id())

coldf = numdf.withColumnRenamed("teams", "teams1")\
    .withColumnRenamed("id", "id1")


numdf.show()
coldf.show()


innerdf = numdf.join(coldf, numdf.id < coldf.id1, "inner")

innerdf.show()

findf = innerdf.selectExpr("concat(teams,' vs ',teams1) as matches")

findf.show()

print("-----SQL SOLUTION-----\n\n\n")


df.createOrReplaceTempView("df_view")

spark.sql("""
          
          select concat(a.teams, ' vs ',b.teams) as matches from 
          df_view a inner join df_view b on a.teams<b.teams

          """).show()
